# Exemple synthetique : retrouver un modele connu

Ce notebook montre un cas volontairement simple : on fixe d'abord un modele `exp_shifted` avec des parametres connus, on genere des concentrations synthetiques a une date donnee, puis on demande a PyAge de retrouver ces parametres a partir des seules observations bruitees.

L'interet pedagogique est direct : la `verite` est connue. On peut donc lire les sorties non seulement comme un resultat numerique, mais comme une verification de ce que fait la calibration.

## Parcours du notebook

Le notebook suit cinq etapes tres courtes :

1. definir le cas synthetique ;
2. comparer les concentrations vraies et les observations bruitees ;
3. lancer la calibration ;
4. lire les trois figures principales ;
5. comparer les parametres vrais et estimes.

`expert_mode = False` par defaut pour rester centre sur l'essentiel.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd().resolve()
while not (ROOT / 'pyage').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

EXAMPLE_DIR = ROOT / 'examples' / 'synthetic' / 'lpm_recovery_single_date'
if str(EXAMPLE_DIR) not in sys.path:
    sys.path.insert(0, str(EXAMPLE_DIR))

from synthetic_case import generate_synthetic_case
from run_lpm_recovery_single_date import main as run_example

expert_mode = False

## 1. Generer le cas synthetique

La verite synthetique est definie dans `generation/generation_settings.yaml`.
Dans cet exemple :

- le modele choisi pour generer les donnees est `exp_shifted` ;
- les traceurs sont `cfc11`, `cfc12` et `cfc113` ;
- une incertitude relative controlee est ajoutee aux concentrations.

La calibration, elle, ne voit ensuite que les observations bruitees et leur colonne `error`.

In [ ]:
case = generate_synthetic_case()
truth = case.truth_payload

truth_table = pd.DataFrame(
    [
        {'parametre': name, 'valeur_vraie': value}
        for name, value in truth['lpm']['parameters'].items()
    ]
)

comparison = case.true_frame[['element', 'concentration']].rename(
    columns={'concentration': 'concentration_vraie'}
).merge(
    case.observed_frame[['element', 'concentration', 'error']].rename(
        columns={'concentration': 'concentration_observee', 'error': 'incertitude'}
    ),
    on='element',
    how='left',
)
comparison['ecart_absolu'] = comparison['concentration_observee'] - comparison['concentration_vraie']
comparison['ecart_relatif_%'] = 100.0 * comparison['ecart_absolu'] / comparison['concentration_vraie']

display(Markdown('**Parametres vrais utilises pour generer les donnees**'))
display(truth_table)

display(Markdown('**Concentrations vraies et observations bruitees**'))
display(comparison.round(3))

display(Markdown(f'Fichier d observations genere : `{case.paths.dataset_path}`'))
display(Markdown(f'Fichier de verite stocke : `{case.paths.truth_path}`'))

Le point important a retenir ici est le suivant : la calibration ne cherche pas a retrouver exactement les observations bruitees point par point. Elle cherche plutot les parametres qui rendent ces observations plausibles compte tenu de l'incertitude imposee.

## 2. Lancer la calibration

Le script ci-dessous :

- regenere les memes observations synthetiques ;
- lance le workflow single-date ;
- reconstruit les trois figures de synthese en y ajoutant la verite synthetique.

Dans cette version d'introduction, on ne garde qu'une seule methode de calibration : `Metropolis_Hastings`.

In [ ]:
results_dir = run_example(force_inline=True)
display(Markdown(f'**Repertoire de resultats :** `{results_dir}`'))
results_dir

## 3. Lire les figures principales

Ces trois figures suffisent pour comprendre la logique complete de l'exemple : donnees, parametres, puis fonction objectif.

In [ ]:
display(Image(filename=str(results_dir / '01_data_model_space.png'), width=900))
display(Markdown("""**Comment lire cette figure**

- le point des observations represente les concentrations effectivement donnees a la calibration ;
- le losange de reference represente le vrai modele synthetique ;
- le nuage de points represente les modeles calibres explores par la methode.

On attend que le nuage calibre se place au voisinage du modele vrai et des observations, sans forcement se confondre parfaitement avec eux a cause du bruit."""))

In [ ]:
display(Image(filename=str(results_dir / '02_parameter_summary.png'), width=900))
display(Markdown("""**Comment lire cette figure**

- chaque histogramme montre la distribution posterieure d un parametre ;
- la ligne verticale de reference marque la valeur vraie utilisee pour generer les donnees.

Le resultat est pedagogiquement bon si la valeur vraie tombe dans la zone dense de la distribution, meme si elle n est pas exactement au maximum."""))

In [ ]:
display(Image(filename=str(results_dir / '03_objective_summary.png'), width=900))
display(Markdown("""**Comment lire cette figure**

- le fond colore represente la fonction objectif calculee sur une grille de parametres ;
- les valeurs les plus faibles correspondent aux zones les plus compatibles avec les donnees ;
- le repere de reference indique les vrais parametres synthetiques.

On cherche ici a verifier que la zone favorable de la fonction objectif recouvre bien le voisinage des parametres vrais."""))

## 4. Comparer parametres vrais et parametres estimes

Le tableau suivant synthetise la recuperation des parametres. Il compare la valeur vraie, la moyenne estimee a posteriori, l ecart-type estime et l ecart entre vrai et estime.

In [ ]:
recovery = pd.read_table(results_dir / 'parameter_recovery_summary.txt', sep='	')
recovery['ecart_relatif_%'] = 100.0 * recovery['difference'] / recovery['true_value']
recovery['dans_1_sigma'] = recovery['difference'].abs() <= recovery['estimated_std']
recovery.round(3)

On ne cherche pas une egalite exacte entre valeur vraie et moyenne estimee. Avec des donnees bruitees, le bon critere de lecture est plutot : la valeur vraie reste-t-elle proche du centre de la distribution estimee, et l ecart observe est-il du meme ordre que l incertitude estimee ?

## 5. Mode expert

Si `expert_mode = True`, le notebook affiche en plus un extrait du nuage posterieur et de la grille de fonction objectif. Cela sert surtout a comprendre la structure interne des sorties, pas a une premiere lecture.

In [ ]:
if expert_mode:
    posterior = pd.read_table(results_dir / 'Metropolis_Hastings' / 'lpm_dist_calibrated.txt', sep='	')
    objective_grid = pd.read_table(results_dir / 'objective_function_grid.txt', sep='	')
    display(Markdown('**Extrait du nuage posterieur**'))
    display(posterior.head())
    display(Markdown('**Extrait de la grille de fonction objectif**'))
    display(objective_grid.head())